In [7]:
import torch
import myneural as mn
import torch.nn.functional as F
import time
torch.manual_seed(42)
text = open('data/tinyshakespeare.txt', 'r').read()
chars = sorted(list(set(text)))
stoi = {s:i for i, s in enumerate(chars)}
itos = {i:s for s, i in stoi.items()}
vocab_size = len(chars)

print(f"Vocab size = {vocab_size}")
def encode(text):
    return [stoi[s] for s in text]

def decode(list):
    return ''.join([itos[i] for i in list])

Vocab size = 65


In [8]:
config = mn.Config()
config.n_embd = 192
config.context_length = 256
config.n_head = 6
config.n_layer = 6
config.batch_size = 64
config.device = 'cuda'
max_iter = 1000
config.vocab_size = vocab_size

In [9]:
data = torch.tensor(encode(text), dtype=torch.long, device=config.device)

n = int(0.9 * len(data))
train = data[:n]
val = data[n:]

def get_batch(split):
    x = train if split == 'train' else val
    ix = torch.randint(0, len(x) - config.context_length, (config.batch_size,), device=config.device)
    xs = torch.stack([x[i:i+config.context_length] for i in ix]) # (B, T)
    ys = torch.stack([x[i+1:i+config.context_length+1] for i in ix]) # (B, T)
    return xs, ys

In [10]:
model = mn.Transformer(config, dtype=torch.float32, device=config.device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [11]:
# # Load the list of tensors
# loaded_params = torch.load("params.pth")

# # Manually re-assign them to your model
# for p, loaded_p in zip(model.parameters(), loaded_params):
#     p.data.copy_(loaded_p.data)

In [12]:
for i in range(max_iter):
    t_start = time.time()
    # forward pass
    xs, ys = get_batch('train')
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        logits, loss = model(xs, targets=ys)
    
    # backward pass
    for p in model.parameters():
        p.grad = None
    loss.backward()
    
    # update
    optimizer.step()
    torch.cuda.synchronize()
    t_end = time.time()
    if i % 10 == 0:
        print(f"Step {i}: Loss = {loss.item()} | {(t_end - t_start)*1000.0:.3f}ms")

Step 0: Loss = 4.751061916351318 | 805.703ms
Step 10: Loss = 3.458918571472168 | 196.243ms
Step 20: Loss = 3.3033361434936523 | 196.279ms
Step 30: Loss = 3.2501649856567383 | 195.339ms
Step 40: Loss = 3.1415858268737793 | 195.898ms
Step 50: Loss = 2.968268632888794 | 197.248ms
Step 60: Loss = 2.7828269004821777 | 189.856ms
Step 70: Loss = 2.690770387649536 | 190.345ms
Step 80: Loss = 2.643418788909912 | 189.946ms
Step 90: Loss = 2.5787739753723145 | 191.043ms
Step 100: Loss = 2.5514743328094482 | 190.984ms
Step 110: Loss = 2.5357494354248047 | 203.222ms
Step 120: Loss = 2.5442819595336914 | 139.795ms
Step 130: Loss = 2.513178825378418 | 147.353ms
Step 140: Loss = 2.503448486328125 | 136.826ms
Step 150: Loss = 2.4961929321289062 | 146.144ms
Step 160: Loss = 2.5123026371002197 | 134.397ms
Step 170: Loss = 2.481243371963501 | 133.127ms
Step 180: Loss = 2.4733028411865234 | 135.066ms
Step 190: Loss = 2.4734997749328613 | 138.104ms
Step 200: Loss = 2.437932252883911 | 137.108ms
Step 210: Lo

In [15]:
params_list = list(model.parameters())
torch.save(params_list, "params.pth")

In [16]:
xs, ys = get_batch('val')

logits, loss = model.forward(xs, ys)
loss

Param(1.6387, device='cuda:0', grad_fn=<AliasBackward0>)

In [17]:
print(''.join(decode(model.generate(torch.zeros((1,1), dtype=torch.long, device=config.device), 2000)[0].tolist())))


MERDERCit:
For tubt help, I grave's some do rot boot:
I put be dament thy first; York Clifford: now I
subject there wrong it that shall becond me of my shile
That be hast of this own beat left
heave of may joy frier the wast
garlest of worships, a but made treasury lord,
That we frob exil boys a dow'st thrage and of better
Than noble of pose fave me you look, while
He avis are tore: but while shall for the fort.

Nursal:
I cafe id you. Then answer, Tow, impeach about,
That you are a mine on and not for a
estraith; king doubting befall me to why fornight:
Ah, prothend in yourself hath in there
To you had at yourself besided in look

DUKE OF HEN WY trange: say, sayt it yeaul,
And the tears to with my is sun,
We with that Rut with peace. Aund it now counsurse
A busicans
This pity foul of Streesir. Your life
That may sal hape, about whed so will.
Look you farse of so thine othere?
To farsh'd the king like flatter die,
And cares; as these by know of day's solden with indeen
Nares with me, 